# Dunnhumby seed49 — M2와 전체결합 예비 확인
**이전 10시드 노트북 대신 이 노트북만 실행하세요. L4 GPU를 선택하세요.**

- Dunnhumby만, **seed49 하나**, M1 → M3+M4-B → 새 M2 → M2+M3+M4-B 총 **4회** 순차 학습합니다. H&M이나 나머지 시드로 자동 확대하지 않습니다.
- 학습 DAY1–704 전체, test DAY705–711. validation·holdout 없음. **300epoch 고정**, 마지막 checkpoint만 평가하며 test 기반 조기종료/최고점 선택은 없습니다.
- 기존 다른 분할 결과는 재사용하지 않습니다. 이번 실행 중 완료 결과/마지막 완료 epoch는 자동 재개합니다.
- 신규상품 추천, train pair를 정답·후보에서 제외, MIN_ITEM_INTER=1, uniform K1.
- CLV 입력은 학습 종료 전365일 관측으로 고정합니다(그래프·BPR 학습은 앞 전체 기간).
- 단일 시드이므로 최종10시드 보고·안정성·유의성·CLV 배정귀속을 주장하지 않습니다. seed49의 과거 모든 실험 미사용을 주장하는 것도 아닙니다.
- 마지막 주는 초기 연구에서 이미 노출된 test입니다. 이번 결과로 같은 test에 맞춰 설계를 수정하면 독립 확증이 아닌 탐색으로 기록해야 합니다. 시드를 늘려도 이 편향이 사라지지 않습니다.

## 구조·판독 기준 유지

M2 사용자 경제표현은 q_C·[T0+(2q_N−1)TN]·b(q_V), 상품은 b(상품 구매금액 백분위). T0=I/TN=0에서 ID와 공동학습하며 N/V/q_C를 모두 유지합니다. ID만 전파하고 경제표현은 내적 전에 연결합니다. 하나의 optimizer, 동결·외부 재정렬 없음.

M2 단독 binary/plain BPR; 전체결합은 중심화 N/V M3 + split N/V M4-B(무효행 raw=1, 학습행 평균 정규화). ID64/2층/batch8192/lr.0005/ID L2.001/rho.25/RBF폭.25, M3 target CV=.20 고정. 추가 구조·하이퍼파라미터 변경 없음.

전체결합의 **전체 사용자 가격·구매금액 가중 적중값@10 및 가중 NDCG@10이 각각 M1과 M3+M4-B를 초과**하고 여섯 Recall/NDCG가 M1의99% 이상인지 **이 시드에서만** 확인합니다. M2 단독 미달로 결합을 중단하지 않습니다. 모든 세그먼트·노출·불리한 결과를 저장합니다. 시드 간 표준편차는 계산 불가로 비워둡니다.

로컬 소형 분할·학습/재개·단일 시드 실행범위·보고 검사 통과. 실제 Drive 전체 데이터/L4 학습은 미실행입니다.

## 1. Drive 연결·고정 코드 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, torch
SOURCE_COMMIT = '6fb7f276c56e9c1888e2f448933556b4c78a46fb'
REPO = Path('/content/clv-lastweek-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. Dunnhumby seed49 설정·준비 — 학습 없음

In [ ]:
import clv_m2_m5_lastweek_test10 as screen
cfg = screen.configure('dunnhumby', seeds=(49,))
assert cfg.dataset == 'dunnhumby' and cfg.seeds == (49,) and cfg.epochs == 300
prepared = screen.prepare(cfg)
assert prepared['protocol']['total_fits'] == 4 and prepared['protocol']['pilot_only']
print('자동재개 경로:', prepared['run_dir'])


## 3. 네 모형 순차 학습
매epoch optimizer/난수상태 저장, 중단 후 재실행 시 재개합니다. loss는25epoch마다 출력되며 그 사이 진행상태는 실행폴더/progress에 기록됩니다. 네 모형 완료 후 종료합니다.

In [ ]:
result = screen.run(cfg, prepared)
assert result['reading']['seed_count'] == 1 and result['reading']['pilot_only']
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 4. 전체 사용자 성능·대응 차이
단일 시드의 절대값/상대변화율입니다. 모든 지표 원본은 ZIP에 저장됩니다.

In [ ]:
metrics = [f'{m}@{k}' for k in (10,20,50) for m in ('recall','ndcg','price_purchase_amount_weighted_hit','vndcg')]
labels = {screen.M1:'M1',screen.BASE:'M3+M4-B',screen.M2:'새 M2',screen.FULL:'M2+M3+M4-B'}
print(result['absolute'].set_index('model_id')[metrics].T.rename(columns=labels).to_string())
comparison = result['comparison'].copy()
comparison['model_id'] = comparison.model_id.map(labels)
comparison['reference'] = comparison.reference.map(labels)
print('\n상대변화율(%)')
print(comparison[comparison.metric.isin(metrics)].pivot(index='metric',columns=['model_id','reference'],values='relative_change_pct').to_string())
print('\nN/V 작동 진단')
print(result['diagnostics'].to_string(index=False))
print(json.dumps(result['paths'],ensure_ascii=False,indent=2))


## 5. 결과 ZIP 다운로드
absolute/summary/comparison/paired_comparison/diagnostics/result.json/protocol.json 포함. checkpoint는 Drive에 남습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
zip_path = Path('/content/m2_m5_lastweek_dunnhumby_seed49_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
    archive.write(prepared['run_dir']/'protocol.json',arcname='protocol.json')
files.download(str(zip_path))
